# RFI development v2 — supported background, excess-based cross flags

**Fix being tested.** In the July 16 00:11–00:31 UTC window, the old absolute cross-coherence gate removes every v0 fitting channel below 117.19 MHz. A converged DPSS solve then produces untrustworthy extrapolation. V1 keeps more individual samples but inherits the absolute-coherence veto and does not distinguish lack of model support from RFI.

V2 makes three changes:
1. **Fit the auto background without an absolute cross-coherence veto.** A high, stable cross correlation is not automatically interference. Initialize with all physically valid auto samples; iteratively exclude positive auto excess.
2. **Detect changes in complex cross correlation relative to its robust, slowly varying time background.** Keep narrowband/band/8–4–2-channel auto detectors. Separate the auto-fitting mask, RFI evidence, and final exclusion mask.
3. **Check model support explicitly.** Evaluate the small tensor-basis information matrix. Flag regions with excessive prediction-variance inflation or strong dependence on regularization; return `NaN` there in the public background model. This is a conditioning check, not proof of a correct continuum or clean data.

**Bounded milestone.** Test the user's early window and the original tail window; execute deterministic solver, support-loss, strong-coherent-background, injected-event, and withheld-row checks. Acceptance requires restored low-frequency support and finite predictions, prompt injected onset detection, preservation of synthetic clean samples, and an explicit veto for deliberately unsupported extrapolation. Real FM cleanliness remains a separate scientific question. No additional campaign scan or package changes.

**Inputs and conventions.** Phase C, raw keys 0 (ground), 4 (air), 04 (ground–air cross); auto in correlator counts, frequency MHz, UTC seconds, true flags mean exclude. Model widths remain 50 ns and 1 mHz. Above 235 MHz is outside the tested domain. All array products remain notebook variables; no campaign product is published. The engineering gate is waived for this exploration.

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, time, warnings
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
from scipy.ndimage import median_filter, maximum_filter
from scipy.linalg import cho_factor, cho_solve, eigh
from scipy.sparse.linalg import LinearOperator, cg
from threadpoolctl import threadpool_limits
from hera_filters.dspec import dpss_operator, fit_solution_matrix
from eigsep_data import MetadataIndex
from IPython.display import display
%matplotlib widget

ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents]
          if (p/'data-analysis/notebooks/arp/marjum-2026-07/rfi_dev_v0.ipynb').exists())
NOTEBOOK_DIR=ROOT/'data-analysis/notebooks/arp/marjum-2026-07'
CAMPAIGN=ROOT/'marjum-2026-07'
blas_limit=threadpool_limits(limits=1)
SPAN, MINUTES = 13, 20
PRIMARY='early'  # 'early' reproduces the reported failure; 'tail' is the original window.
CASES={'early':dict(offset_s=4*3600), 'tail':dict(offset_s=None)}
BAND_MHZ=(35.,250.)
TESTED_BAND_MHZ=(35.,235.)
REPORT_BANDS={
    '50-88 (DTV-lo)':(50.,88.), '88-108 (FM)':(88.,108.),
    '108-137':(108.,136.5), '137-138 (Orbcomm)':(136.5,138.5),
    '138-174':(138.5,174.), '174-216 (DTV-hi)':(174.,216.),
    '216-235':(216.,235.)}
P=dict(freq_halfwidth_s=50e-9,time_halfwidth_hz=1e-3,eigenvalue_cutoff=1e-12,
       fit_clip=8.,point_cut=6.,other_point_cut=8.,negative_cut=8.,
       group_cut=6.,group_cell_cut=2.,group_fraction=.35,tile_channels=8,
       comb_cut=6.,comb_tooth_cut=2.,comb_fraction=.10,comb_min_teeth=16,
       ridge=1e-6,cg_rtol=1e-8,cg_maxiter=300,robust_rounds=50,mask_change_tol=5e-4,
       cross_cut=8.,cross_clip=8.,cross_iterations=80,cross_step_tol=1e-3,
       support_inflation=5.,prior_fraction=.10,refit_rounds=2,
       time_guard=1,frequency_guard=0)
display(pd.Series(P,name='parameters'))

## Selection and provenance — regenerated for each case

The window is selected using `time_best`; printed filenames are close times. Reject phase, clock, cadence, accumulation or RF-switch changes, declared SNAP test flips, all-zero moves, box-air outages and mux tuning. Retain RFI and thunderstorm intervals for the experiment. The early window lies in the archive's thunderstorm interval; that fact alone does not establish the source of any feature.

Campaign references: `marjum-2026-07/INDEX.md`, `data/README.md`, and the mask declarations in `curation/select_files.py`. The archive distinguishes grid-locked 8-channel structure from a walking near-1-MHz transmitter comb. Here 8/4/2 are pattern hypotheses, not source attribution.

In [ ]:
import runpy
catalog=runpy.run_path(str(CAMPAIGN/'curation/select_files.py'))
idx=MetadataIndex(CAMPAIGN/'data')
selection=idx.select()
meta=selection.meta.assign(span=selection.visits(gap_s=600))
spans=meta.groupby('span').agg(start=('time_best','min'),end=('time_best','max'),rows=('file','size'))
spans[['start','end']]=spans[['start','end']].apply(pd.to_datetime,unit='s',utc=True)
display(spans)

def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def git_state(path):
    def git(*a):return subprocess.check_output(['git','-C',str(path),*a],text=True).strip()
    return dict(commit=git('rev-parse','HEAD'),branch=git('branch','--show-current'),
                dirty=bool(git('status','--porcelain','--untracked-files=normal')))

def load_case(name,offset_s):
    t0=spans.loc[SPAN,'start'].timestamp();t1=spans.loc[SPAN,'end'].timestamp()
    lo=t1-MINUTES*60 if offset_s is None else t0+offset_s
    hi=lo+MINUTES*60
    w=idx.select(time=(lo,hi))
    assert len(w.meta)>0 and set(w.meta.filter_phase)=={'C'}
    assert w.meta.sync_consistent.all(), 'Clock estimates: use a different window.'
    assert set(w.meta.rfswitch)=={'RFANT'}, 'Split on RF switch changes.'
    assert w.meta.integration_time.nunique()==1, 'Split accumulation changes.'
    checks=[(x['name'],x['start'],x['end']) for x in catalog['MANDATORY_MASKS']]
    checks += [(x['name'],a,b) for x in catalog['CONDITIONAL_MASKS']
               if x['name'] in ('box-air-outage','mux-tuning') for a,b in x.get('windows',[])]
    for label,a,b in checks:
        assert not w.meta.time_best.between(pd.Timestamp(a).timestamp(),pd.Timestamp(b).timestamp()).any(),label
    assert not set(w.meta.file)&catalog['CORRUPT_FILES']
    bundles={key:w.load_bundle(key=key,band_mhz=BAND_MHZ,missing='raise') for key in ('0','4','04')}
    b=bundles['4']
    for v in bundles.values():
        assert np.array_equal(v.t,b.t) and np.array_equal(v.freqs_mhz,b.freqs_mhz)
        assert v.meta[['file','row']].equals(b.meta[['file','row']])
    t,f=b.t,b.freqs_mhz
    dt=b.meta.integration_time.to_numpy(float);df=np.diff(f).mean()
    assert np.allclose(np.diff(t),dt[0],rtol=1e-5)
    assert np.allclose(np.diff(f),df)
    ch=np.rint(f/df).astype(int);assert np.allclose(ch*df,f)
    d=np.asarray(b.data,float);g=np.asarray(bundles['0'].data,float);x=bundles['04'].data
    rt=np.sqrt(2*dt[:,None]*df*1e6)
    domain=(f>=TESTED_BAND_MHZ[0])&(f<TESTED_BAND_MHZ[1])
    valid=np.isfinite(d)&np.isfinite(g)&np.isfinite(x)&(d>0)&(g>0)&domain
    cx=np.divide(x,np.sqrt(np.maximum(d,1)*np.maximum(g,1)))*rt
    files=sorted(w.meta.file.unique())
    prov=dict(name=name,start_utc=str(pd.to_datetime(t[0],unit='s',utc=True)),
              end_utc=str(pd.to_datetime(t[-1],unit='s',utc=True)),phase='C',keys=['0','4','04'],
              integration_s=float(dt[0]),df_mhz=float(df),shape=list(d.shape),
              file_sha256={fn:sha(CAMPAIGN/'data'/fn) for fn in files})
    print(name,prov['start_utc'],'to',prov['end_utc']);print('\n'.join(files))
    return dict(name=name,D=d,G=g,X=x,t=t,f=f,channel=ch,rt=rt,valid=valid,
                domain=domain,cross_z=cx,provenance=prov)

cases={name:load_case(name,**cfg) for name,cfg in CASES.items()}
import eigsep_data.bundle, eigsep_data.index, hera_filters.dspec
source=json.loads((NOTEBOOK_DIR/'rfi_dev_v2.ipynb').read_text())
provenance=dict(analysis=git_state(ROOT/'data-analysis'),campaign=git_state(CAMPAIGN),
    parameters=P,source_v1_sha256=sha(NOTEBOOK_DIR/'rfi_dev_v1.ipynb'),
    notebook_code_sha256=hashlib.sha256('\n'.join(''.join(c['source']) for c in source['cells'] if c['cell_type']=='code').encode()).hexdigest(),
    catalog_sha256=sha(CAMPAIGN/'curation/select_files.py'),
    installed={p:version(p) for p in ['numpy','scipy','hera_filters','eigsep_data']},
    installed_source={m.__name__:dict(path=m.__file__,sha256=sha(m.__file__))
                      for m in [eigsep_data.bundle,eigsep_data.index,hera_filters.dspec]},
    cases={name:a['provenance'] for name,a in cases.items()})
display(provenance)

## Smooth auto model and design support

The auto model is `A_t @ C @ A_f.T`, with the same DPSS space as v1. A fixed robust spectrum scales the least-squares residuals; QR changes basis coordinates, not the physical model space. CG applies separable projections, with two small marginal preconditioners per solve. No factorization depends on an individual time row.

The initial eligible set contains **all valid autos**, regardless of absolute cross coherence. Positive-excess clipping retains lower samples while the model settles, and reconsiders eligible samples each round. This can still select a low envelope or fail to converge where the continuum model is wrong; report the iteration history and do not call low residuals evidence of an unbiased fit.

For the final fitting mask, form the small information matrix `H = B.T W B`, where `B = A_t ⊗ A_f` in the orthonormal, scaled coordinates. Compute its inverse with the same ridge as the fit. Prediction standard-error inflation is the square root of the ratio of `diag(B (H+ridge I)^-1 B.T)` to the fully observed design variance. A second diagnostic measures the fraction attributable to ridge stabilization. Reject inflation >5 or ridge fraction >10%. These thresholds are engineering safeguards, not calibrated physical error bars. The computation forms a basis-size matrix (297×297 for these windows), not a sample-size matrix.

In [ ]:
def bases(times, freqs):
    af = dpss_operator(freqs*1e6, [0], [P['freq_halfwidth_s']],
                       eigenval_cutoff=[P['eigenvalue_cutoff']])[0].real
    at = dpss_operator(times-times[0], [0], [P['time_halfwidth_hz']],
                       eigenval_cutoff=[P['eigenvalue_cutoff']])[0].real
    return at, af


class TensorFit:
    def __init__(self, times, freqs, data):
        at, af = bases(times, freqs)
        self.qt = np.linalg.qr(at)[0]
        self.scale = np.maximum(median_filter(np.median(np.maximum(data,1),axis=0),size=31),1e4)
        self.qf = np.linalg.qr(af/self.scale[:,None])[0]
        self.shape = (self.qt.shape[1], self.qf.shape[1])
        self.history = []

    def solve(self, data, keep):
        start = time.perf_counter()
        qt,qf = self.qt,self.qf
        w = np.asarray(keep,float)
        assert w.sum() > np.prod(self.shape), 'Insufficient fitting samples.'
        y = np.where(keep, data/self.scale, 0.)  # Invalid values must not enter 0*NaN.
        rhs = qt.T @ y @ qf
        ridge = P['ridge']
        def matvec(a):
            a = a.reshape(self.shape)
            return (qt.T @ (w*(qt @ a @ qf.T)) @ qf + ridge*a).ravel()
        op = LinearOperator((rhs.size,rhs.size),matvec=matvec,dtype=float)
        ct = cho_factor(qt.T @ (w.mean(1)[:,None]*qt)/w.mean() + ridge*np.eye(self.shape[0]))
        cf = cho_factor(qf.T @ (w.mean(0)[:,None]*qf) + ridge*np.eye(self.shape[1]))
        def precondition(a):
            return cho_solve(cf,cho_solve(ct,a.reshape(self.shape)).T).T.ravel()
        pre = LinearOperator(op.shape,matvec=precondition,dtype=float)
        iterations = []
        coeff,info = cg(op,rhs.ravel(),M=pre,rtol=P['cg_rtol'],maxiter=P['cg_maxiter'],
                        callback=lambda _:iterations.append(1))
        rel = np.linalg.norm(matvec(coeff)-rhs.ravel()) / max(np.linalg.norm(rhs),1e-30)
        record = dict(seconds=time.perf_counter()-start, iterations=len(iterations),
                      info=int(info), relative_normal_residual=float(rel), factorizations=2,
                      kept_fraction=float(w.mean()))
        self.history.append(record)
        if info != 0: raise RuntimeError(f'CG did not converge: {record}')
        return (qt @ coeff.reshape(self.shape) @ qf.T)*self.scale

def residual_z(data, model, normalization):
    return np.where(model > 0, (data/np.maximum(model,1)-1)*normalization, np.inf)

def robust_model(data, eligible, solver, normalization):
    keep = eligible.copy()
    trace = []
    for iteration in range(P['robust_rounds']):
        model = solver.solve(data, keep)
        z = residual_z(data,model,normalization)
        # Always reconsider eligible lower samples; no monotone loss of initial FM gaps.
        updated = eligible & np.isfinite(z) & (z < P['fit_clip'])
        change = np.mean(updated != keep)
        trace.append(dict(round=iteration, changed_fraction=change, kept=updated.mean()))
        keep = updated
        if change < P['mask_change_tol']: break
    # Ensure the returned model corresponds exactly to the returned training mask.
    model = solver.solve(data,keep)
    return model,keep,pd.DataFrame(trace)

In [ ]:
def support(solver,keep):
 start=time.perf_counter();qt,qf=solver.qt,solver.qf;nt,nf=solver.shape
 tt=np.einsum('ta,tb->abt',qt,qt).reshape(nt*nt,-1)
 moments=(tt@keep.astype(float)).reshape(nt,nt,-1)
 h=np.einsum('abf,fi,fj->aibj',moments,qf,qf,optimize=True).reshape(nt*nf,nt*nf)
 ev,v=eigh((h+h.T)*.5);ev=np.maximum(ev,0)
 inv=(v/(ev+P['ridge']))@v.T
 inv2=(v/(ev+P['ridge'])**2)@v.T
 def diag(cov):
  fc=np.einsum('aibj,fi,fj->abf',cov.reshape(nt,nf,nt,nf),qf,qf,optimize=True)
  return (tt.T@fc.reshape(nt*nt,-1))
 var=np.maximum(diag(inv),1e-30)
 nominal=(qt**2).sum(1)[:,None]*(qf**2).sum(1)[None,:]
 infl=np.sqrt(var/nominal);prior=np.clip(P['ridge']*diag(inv2)/var,0,1)
 return infl,prior,time.perf_counter()-start



def legacy_v0(a,train_rows=None):
    at,af=bases(a['t'],a['f'])
    # Same comparison semantics as v0, including nonpositive-auto NaNs.
    with np.errstate(invalid='ignore',divide='ignore'):
        z=np.abs(a['X'])/(np.sqrt(a['D'])*np.sqrt(a['G'])).clip(1,np.inf)*a['rt']
    point=~(z>20);mf=point.mean(0)>.9;mt=np.all(point[:,mf],axis=1)
    if train_rows is not None:mt &= train_rows
    start=time.perf_counter()
    if not mf.any() or not mt.any():
        model=np.full_like(a['D'],np.nan);condition=np.inf
    else:
        condition=np.linalg.cond(af.T@(mf[:,None]*af))
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',UserWarning)
            ff=fit_solution_matrix(mf.astype(float),af)
            ft=fit_solution_matrix(mt.astype(float),at)
        model=at@(ft@((af@(ff@a['D'].T)).T*mt[:,None]))
    return dict(model=model,mf=mf,mt=mt,condition=condition,seconds=time.perf_counter()-start)

# Independent small-system check, including the information-matrix contraction.
rng=np.random.default_rng(912)
s=TensorFit.__new__(TensorFit)
s.qt=np.linalg.qr(rng.normal(size=(31,4)))[0];s.qf=np.linalg.qr(rng.normal(size=(27,5)))[0]
s.scale=np.ones(27);s.shape=(4,5);s.history=[]
y=rng.normal(size=(31,27));w=rng.random(y.shape)>.35
pred=s.solve(y,w)
B=np.einsum('ti,fj->tfij',s.qt,s.qf).reshape(y.size,-1)
H=B[w.ravel()].T@B[w.ravel()]
coef=np.linalg.solve(H+P['ridge']*np.eye(20),B[w.ravel()].T@y.ravel()[w.ravel()])
np.testing.assert_allclose(pred,(B@coef).reshape(y.shape),atol=1e-7)
infl,prior,_=support(s,w)
inv=np.linalg.inv(H+P['ridge']*np.eye(20))
expected=np.sqrt(np.einsum('ni,ij,nj->n',B,inv,B)/np.sum(B*B,axis=1)).reshape(y.shape)
np.testing.assert_allclose(infl,expected,rtol=1e-8)
print('Dense weighted solver and support-contraction checks passed.')

## Cross changes and event detectors

Normalize the **complex** cross as in v0, but fit its slowly varying real/imaginary background using only the time basis, independently at each frequency. Huber IRLS reuses one time basis and forms all small weighted systems in a single tensor contraction. The cross fit uses batched 9×9 solves per frequency, rather than the expensive frequency-basis fits per time row. Its system count and elapsed time are reported separately from the auto solver; no explicit matrix inverses are formed. The maximum update must fall below 0.001 in v0 z units, otherwise execution stops. A robust residual scale (floor 1 in v0 z units) gives a cross-change score. There is no absolute-coherence flag.

Auto point thresholds remain 6 in the supplied RFI bands, 8 elsewhere. Band pooling and comb phase tests are retained from v1. Detected comb teeth are removed from band pooling, preserving inter-tooth gaps. Calibration statistics use training rows only. Time guards are applied only to the final output, so held-out measurements cannot affect neighboring training exclusions.

Cross-only excursions flag the final data, but exclude the air auto from fitting only when it also shows >2 units of positive auto z. A changing cross need not imply a contaminated auto. All other event reasons used for refitting come from the air auto. Support rejection is an output safeguard and never preemptively deletes scarce training anchors. Two event-mask refits are used after the converged robust initialization; the final detector can flag additional samples that were still present in the last fit. Both masks are exposed, and held-out predictions test the resulting model.

In [ ]:
def cross_background(cz,qt,reference_valid):
    start=time.perf_counter()
    usable=reference_valid & np.isfinite(cz)
    active=np.any(usable,axis=0)
    if not active.any(): raise ValueError('No valid cross-background training samples.')
    good=usable[:,active];data=np.where(good,cz[:,active],0.)
    rank=qt.shape[1]
    # Time-only normal matrices are batched across frequency: rank is 9 here.
    qt_outer=np.einsum('ta,tb->abt',qt,qt).reshape(rank*rank,-1)
    coeff=qt.T@data;fitted=qt@coeff
    history=[]
    for _ in range(P['cross_iterations']):
        residual=data-fitted
        weights=good*np.minimum(1.,P['cross_clip']/np.maximum(abs(residual),1e-12))
        rhs=qt.T@(weights*data)
        gram=(qt_outer@weights).reshape(rank,rank,-1).transpose(2,0,1)
        gram += P['ridge']*np.eye(rank)[None,:,:]
        coeff=np.linalg.solve(gram,rhs.T[:,:,None])[:,:,0].T
        updated=qt@coeff
        change=float(np.max(abs(updated-fitted)))
        history.append(change);fitted=updated
        if change<P['cross_step_tol']:break
    converged=history[-1]<P['cross_step_tol']
    if not converged:
        raise RuntimeError(f'Cross-background robust solve did not converge: last step {history[-1]:.4g}; inspect data and cross_iterations.')
    model=np.zeros_like(cz);model[:,active]=fitted
    residual=cz-model
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        sr=1.4826*np.nanmedian(np.where(usable,abs(residual.real),np.nan),axis=0)
        si=1.4826*np.nanmedian(np.where(usable,abs(residual.imag),np.nan),axis=0)
    scale=np.maximum(np.nan_to_num(np.maximum(sr,si),nan=1.),1.)
    score=abs(residual)/scale
    return model,score,dict(iterations=len(history),last_step=history[-1],
                            reached_step_tolerance=converged,systems_solved=int(active.sum()*len(history)),
                            system_rank=rank,seconds=time.perf_counter()-start)

In [ ]:
def temporal_center_scale(z, usable):
    a = np.where(usable & np.isfinite(z),z,np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        center = np.nanmedian(a,axis=0)
        scale = 1.4826*np.nanmedian(np.abs(a-center),axis=0)
    return np.nan_to_num(center), np.maximum(np.nan_to_num(scale,nan=1.),1.)

def detect(data, model, cross_score, valid, normalization, freqs, channels, reference_valid=None, guard=False):
    reference_valid = valid if reference_valid is None else reference_valid & valid
    z = residual_z(data,model,normalization)
    point_threshold = np.full(len(freqs),P['other_point_cut'])
    for lo,hi in REPORT_BANDS.values():
        point_threshold[(freqs>=lo)&(freqs<hi)] = P['point_cut']
    reasons = {'invalid_or_outside_domain': ~valid,
               'cross_change': cross_score > P['cross_cut'],
               'positive_excess': z > point_threshold,
               'negative_or_model_failure': (z < -P['negative_cut']) | ~np.isfinite(z)}
    center,scale = temporal_center_scale(z, reference_valid & (z<P['fit_clip']))
    u = np.nan_to_num((z-center)/scale,nan=0.,posinf=0.,neginf=0.)

    finite_z = np.where(np.isfinite(z),z,0.)
    curvature = finite_z - .5*(np.roll(finite_z,1,axis=1)+np.roll(finite_z,-1,axis=1))
    curvature = np.nan_to_num(curvature,nan=0.,posinf=0.,neginf=0.) / np.sqrt(1.5)
    # Never use wraparound neighbours or the untested low-count edge as evidence.
    evidence = valid & np.isfinite(z)
    evidence &= np.roll(evidence,1,axis=1) & np.roll(evidence,-1,axis=1)
    evidence[:,[0,-1]] = False
    evidence &= ~((freqs>=88)&(freqs<108))[None,:]
    _,cs = temporal_center_scale(curvature,evidence & reference_valid)
    tooth = curvature/cs
    comb_scores = np.zeros((len(data),8)); phase_on=np.zeros_like(comb_scores,dtype=bool)
    regions = [(35,108),(108,174),(174,235)]
    for phase in range(8):
        j = channels%8 == phase
        good = evidence[:,j]; count=good.sum(1)
        score=(np.clip(tooth[:,j],-3,8)*good).sum(1)/np.sqrt(np.maximum(count,1))
        hits=(tooth[:,j]>P['comb_tooth_cut'])&good
        coverage=np.ones(len(data),dtype=bool)
        for lo,hi in regions:
            region=(freqs[j]>=lo)&(freqs[j]<hi)
            coverage &= hits[:,region].sum(1)>=2
        on=(score>P['comb_cut']) & (hits.sum(1)>=np.maximum(4,P['comb_fraction']*count))
        on &= coverage & (count>=P['comb_min_teeth'])
        comb_scores[:,phase]=score; phase_on[:,phase]=on
    comb_events={}
    for spacing in (8,4,2):
        for phase in range(spacing):
            comb_events[f'{spacing}:{phase}']=np.all(phase_on[:,phase::spacing],axis=1)
    comb = phase_on[:,channels%8]
    reasons['comb_event']=comb
    bands = np.zeros_like(valid)
    band_scores,band_events = {},{}
    for label,(lo,hi) in REPORT_BANDS.items():
        js = np.flatnonzero((freqs>=lo)&(freqs<hi))
        groups = [(label,js)] + [(f'{label}/tile{k}',js[k:k+P['tile_channels']])
                               for k in range(0,len(js),P['tile_channels'])]
        for name,j in groups:
            if len(j)<3: continue
            good = valid[:,j] & ~comb[:,j]
            count = good.sum(1)
            raw = (np.clip(u[:,j],-3,6)*good).sum(1)/np.sqrt(np.maximum(count,1))
            ref_rows = np.any(reference_valid[:,j],axis=1)
            ref = raw[ref_rows]
            med = np.median(ref) if len(ref) else 0.
            sigma = max(1.,1.4826*np.median(np.abs(ref-med))) if len(ref) else 1.
            score = (raw-med)/sigma
            fraction = ((u[:,j]>P['group_cell_cut'])&good).sum(1)/np.maximum(count,1)
            on = (score>P['group_cut']) & (fraction>=P['group_fraction']) & (count>=3)
            bands[np.ix_(on,j)] = True
            band_scores[name],band_events[name] = score,on
    reasons['band_event'] = bands
    for reason in reasons:
        if reason != 'invalid_or_outside_domain': reasons[reason] &= valid
    unguarded=np.logical_or.reduce(list(reasons.values()))
    if guard:
        expanded=maximum_filter(unguarded,size=(2*P['time_guard']+1,2*P['frequency_guard']+1),mode='constant')
        reasons['guard']=expanded & ~unguarded
    flag=np.logical_or.reduce(list(reasons.values()))
    return dict(flag=flag,z=z,reasons=reasons,band_scores=band_scores,band_events=band_events,
                comb_scores=comb_scores,comb_events=comb_events,phase_on=phase_on)

In [ ]:
def pipeline(a,train_rows=None):
    start=time.perf_counter()
    data,t,f,rt,valid,cz,ch=[a[k] for k in ('D','t','f','rt','valid','cross_z','channel')]
    train_rows=np.ones(len(t),bool) if train_rows is None else np.asarray(train_rows,bool)
    reference=valid & train_rows[:,None]
    solver=TensorFit(t,f,data[train_rows])  # Numerical scale never sees held-out auto values.
    model,robust_keep,trace=robust_model(data,reference,solver,rt)
    cbg,cscore,cstats=cross_background(cz,solver.qt,reference)
    refit_trace=[]
    keep=robust_keep
    for iteration in range(P['refit_rounds']):
        d=detect(data,model,cscore,valid,rt,f,ch,reference_valid=reference)
        reasons=d['reasons']
        reject=(reasons['positive_excess']|reasons['negative_or_model_failure']|
                reasons['band_event']|reasons['comb_event']|
                (reasons['cross_change']&(d['z']>2)))
        updated=robust_keep & ~reject
        change=float(np.mean(updated!=keep));keep=updated
        model=solver.solve(data,keep)
        refit_trace.append(dict(round=iteration,changed_fraction=change,kept_fraction=keep.mean()))
        if change<P['mask_change_tol']:break
    infl,prior,support_seconds=support(solver,keep)
    supported=(infl<=P['support_inflation'])&(prior<=P['prior_fraction'])&(model>0)&np.isfinite(model)&a['domain']
    d=detect(data,model,cscore,valid & supported,rt,f,ch,reference_valid=reference & supported)
    # Preserve independent validity, support, and cross-change explanations.
    d['reasons']['invalid_or_outside_domain']=~valid
    d['reasons']['unsupported_background']=~supported
    d['reasons']['cross_change']=(cscore>P['cross_cut'])&valid
    rfi=np.logical_or.reduce([d['reasons'][key] for key in
                              ('positive_excess','cross_change','band_event','comb_event')])
    before_guard=np.logical_or.reduce(list(d['reasons'].values()))
    expanded=maximum_filter(before_guard,size=(2*P['time_guard']+1,2*P['frequency_guard']+1),mode='constant')
    d['reasons']['guard']=expanded & ~before_guard
    d.update(flag=expanded,rfi_flag=rfi,model_raw=model,model=np.where(supported,model,np.nan),
             model_support_ok=supported,inflation=infl,prior_fraction=prior,fit_keep=keep,
             z=np.where(supported,d['z'],np.nan),cross_background=cbg,cross_score=cscore,
             cross_stats=cstats,solver=solver,robust_trace=trace,refit_trace=pd.DataFrame(refit_trace),
             support_seconds=support_seconds,seconds=time.perf_counter()-start)
    return d

legacy={name:legacy_v0(a) for name,a in cases.items()}
results={}
for name,a in cases.items():
    results[name]=pipeline(a)
    r=results[name]
    print(name,'seconds',r['seconds'],'robust rounds',len(r['robust_trace']),
          'last mask change',r['robust_trace'].iloc[-1].changed_fraction,
          'support calculation seconds',r['support_seconds'])
    print('Cross fit:',r['cross_stats'])
    display(r['robust_trace'].tail(5));display(r['refit_trace'])

# Main interactive outputs. True means exclude. Unsupported model values are NaN.
a=cases[PRIMARY];result=results[PRIMARY]
D,G,X,t,f,channel,sqrt_n=[a[k] for k in ('D','G','X','t','f','channel','rt')]
flag_v2=result['flag'];rfi_flag_v2=result['rfi_flag']
model_v2=result['model'];model_support_ok=result['model_support_ok']
fit_keep_v2=result['fit_keep']

## Real-window results

The old v0 model is shown raw to expose its extrapolation failure; it is not an acceptable reference truth. V2's public model is blank where support fails. “Retained” includes all final exclusion reasons, while “RFI evidence” excludes support/validity failures. Support and retention fractions do not establish unbiased FM recovery.

In [ ]:
report=[]
for name,a in cases.items():
    r=results[name];old=legacy[name]
    first=a['f'][old['mf']][0] if old['mf'].any() else np.nan
    print(name,'first v0 fitted MHz:',first,'v0 frequency normal-matrix condition:',old['condition'])
    for band,(lo,hi) in {'35-50':(35.,50.),**REPORT_BANDS}.items():
        j=(a['f']>=lo)&(a['f']<hi)
        report.append(dict(case=name,band=band,
            absolute_cross_gt20=float((abs(a['cross_z'][:,j])>20).mean()),
            v0_fit_fraction=float((old['mt'][:,None]&old['mf'][None,:])[:,j].mean()),
            v0_nonpositive_model=float((old['model'][:,j]<=0).mean()),
            v2_fit_fraction=float(r['fit_keep'][:,j].mean()),
            v2_supported=float(r['model_support_ok'][:,j].mean()),
            v2_retained=float((~r['flag'][:,j]).mean()),
            v2_rfi_evidence=float(r['rfi_flag'][:,j].mean()),
            v2_inflation_p90=float(np.quantile(r['inflation'][:,j],.9))))
report=pd.DataFrame(report);display(report)
for name,a in cases.items():
    r=results[name]
    print(name,'overlapping reason fractions within 35-235 MHz:')
    display(pd.Series({k:v[:,a['domain']].mean() for k,v in r['reasons'].items()}))
    print('Distinct fitting row masks:',len(np.unique(np.packbits(r['fit_keep'],axis=1),axis=0)),
          'small preconditioner factorizations:',sum(x['factorizations'] for x in r['solver'].history),
          'one information-matrix eigendecomposition; v0 fit seconds:',legacy[name]['seconds'])
    print('Cross-background batched systems:',r['cross_stats']['systems_solved'], 'rank:',r['cross_stats']['system_rank'], 'seconds:',r['cross_stats']['seconds'])

for name,a in cases.items():
    r=results[name];f=a['f'];minutes=(a['t']-a['t'][0])/60
    extent=[f[0],f[-1],minutes[-1],0]
    fig,axs=plt.subplots(2,3,figsize=(15,8),sharex=True,sharey=True)
    for ax,image,title in zip(axs[0],[a['D'],legacy[name]['model'],r['model']],
                             ['Air data / counts','v0 raw model (untrusted)','v2 supported model / counts']):
        im=ax.imshow(image,aspect='auto',extent=extent,norm=LogNorm(1e4,1e7),interpolation='nearest')
        ax.set_title(title)
    fig.colorbar(im,ax=list(axs[0]),label='Counts',shrink=.8)
    im=axs[1,0].imshow(r['inflation'],aspect='auto',extent=extent,norm=LogNorm(1,30),interpolation='nearest')
    axs[1,0].set_title('Design noise amplification (limit 5)');fig.colorbar(im,ax=axs[1,0])
    axs[1,1].imshow(r['flag'],aspect='auto',extent=extent,cmap='gray_r',interpolation='nearest')
    axs[1,1].set_title('Final flag: black = excluded')
    im=axs[1,2].imshow(np.where(r['flag'],np.nan,r['z']),aspect='auto',extent=extent,
                      cmap='RdBu_r',vmin=-8,vmax=8,interpolation='nearest')
    axs[1,2].set_title('Retained signed auto z');fig.colorbar(im,ax=axs[1,2])
    for ax in axs.flat:ax.axvline(235,color='gray',ls=':',lw=.8)
    for ax in axs[1]:ax.set_xlabel('MHz')
    for ax in axs[:,0]:ax.set_ylabel('Minutes')
    fig.suptitle(name+' — '+a['provenance']['start_utc']);plt.show()

fig,axs=plt.subplots(2,2,figsize=(14,8))
for row,(name,a) in enumerate(cases.items()):
    r=results[name];f=a['f'];j=(f>=35)&(f<108)
    axs[row,0].plot(f[j],np.median(a['D'],axis=0)[j],label='Data median',color='k')
    axs[row,0].plot(f[j],np.median(r['model_raw'],axis=0)[j],label='v2 raw model median')
    axs[row,0].set_yscale('log');axs[row,0].set_ylabel(name+' / counts');axs[row,0].legend()
    axs[row,1].plot(f[j],r['fit_keep'][:,j].mean(0),label='Fitting support')
    axs[row,1].plot(f[j],r['model_support_ok'][:,j].mean(0),label='Supported model')
    axs[row,1].plot(f[j],(~r['flag'][:,j]).mean(0),label='Retained data')
    axs[row,1].set_ylim(0,1.03);axs[row,1].legend()
for ax in axs.flat:ax.set_xlabel('MHz');ax.axvspan(88,108,color='gray',alpha=.12)
fig.tight_layout();plt.show()

## Held-out rows — full v2 pipeline, without selection on its prediction residual

Withhold every 37th integration and a central 20-second block. Numerical scaling, auto-fit mask updates, cross-background estimation and detector calibration use training rows only. The full final model predicts withheld rows; report the supported fraction as well as residuals. For a common, model-independent diagnostic subset, evaluate withheld data whose auto counts lie within four training-derived MADs of their per-channel training median. This selects the temporal core but cannot remove persistent RFI; metrics are conditional prediction checks, not proof of a sky continuum. No threshold is chosen using the held-out results.

In [ ]:
holdout_rows=[];holdout_runs={}
for name,a in cases.items():
    t=a['t'];hold=(np.arange(len(t))%37==0)|((t-t[0]>590)&(t-t[0]<610))
    h=pipeline(a,train_rows=~hold);holdout_runs[name]=h
    old=legacy_v0(a,train_rows=~hold)
    core_median=np.median(a['D'][~hold],axis=0)
    core_mad=np.maximum(1.4826*np.median(abs(a['D'][~hold]-core_median),axis=0),core_median/a['rt'][0,0])
    core=abs(a['D']-core_median)<=4*core_mad
    zold=residual_z(a['D'],old['model'],a['rt'])
    znew=residual_z(a['D'],h['model_raw'],a['rt'])
    for band,(lo,hi) in {'35-50':(35.,50.),**REPORT_BANDS}.items():
        j=(a['f']>=lo)&(a['f']<hi)
        candidate=hold[:,None]&a['valid']&core&j[None,:]
        common=candidate & h['model_support_ok']
        for label,z in [('v0',zold),('v2',znew)]:
            vals=z[common];finite=np.isfinite(vals);q=vals[finite]
            med=np.median(q) if len(q) else np.nan
            holdout_rows.append(dict(case=name,band=band,model=label,candidates=int(candidate.sum()),
                supported_fraction=float(common.sum()/max(candidate.sum(),1)),
                finite_prediction_fraction=float(finite.mean()) if len(vals) else np.nan,
                median_z=med,mad_z=float(1.4826*np.median(abs(q-med))) if len(q) else np.nan,
                p90_abs_z=float(np.quantile(abs(q),.9)) if len(q) else np.nan))
holdout_table=pd.DataFrame(holdout_rows);display(holdout_table)

## Known-truth checks: strong coherent background, RFI onsets, and missing support

The deterministic simulation has a smooth auto continuum with independent radiometer noise. Its cross has **60 units of stable coherence**, plus a slow complex drift and frequency-dependent phase: the original absolute threshold would reject virtually everything. V2 must retain the clean data. Add persistent FM lines, weak 4-sigma 8/4/2-channel combs, a band burst, a narrow line, a broadband impulse, and a cross-only change. First-integration and event recall count **actual RFI evidence only**, excluding time guards, validity failures and unsupported-model vetoes. Synthetic physics is simplified; success does not validate real FM recovery.

A separate test removes all training data below 117.2 MHz to reproduce the dangerous extrapolation geometry. The support diagnostic must reject that region. This is an intentionally broken training mask, not a suggested flagging strategy.

In [ ]:
rng=np.random.default_rng(20260918)
f=cases[PRIMARY]['f'];ch=cases[PRIMARY]['channel'];domain=cases[PRIMARY]['domain']
ts=np.linspace(0,1200,512);rt=np.full((len(ts),1),cases[PRIMARY]['rt'][0,0])
truth=1e6*(1+.12*np.cos(2*np.pi*f[None,:]/150))*(1+.02*np.cos(2*np.pi*ts[:,None]/1800))
clean=truth*(1+rng.normal(size=truth.shape)/rt)
noise=(rng.normal(size=truth.shape)+1j*rng.normal(size=truth.shape))/np.sqrt(2)
coherent=60*np.exp(1j*(2*np.pi*f[None,:]/9+.35*np.sin(2*np.pi*ts[:,None]/1800)))
cz=noise+coherent
sv=np.broadcast_to(domain,truth.shape).copy()
synthetic=dict(name='synthetic',D=clean,t=ts,f=f,channel=ch,rt=rt,valid=sv,domain=domain,cross_z=cz)
null=pipeline(synthetic)
weak_null=pipeline({**synthetic,'cross_z':noise})
injected=clean.copy();cx=cz.copy();truth_rfi=np.zeros_like(sv);events=[]
fm=(f>=88)&(f<108)
def inject(name,start,stop,cols,sigma,cross_sigma=0):
    j=np.flatnonzero(cols);rows=np.arange(start,stop)
    injected[np.ix_(rows,j)] += sigma*truth[np.ix_(rows,j)]/rt[start:stop]
    cx[np.ix_(rows,j)] += cross_sigma
    truth_rfi[np.ix_(rows,j)]=True
    events.append(dict(name=name,start=start,stop=stop,columns=j,sigma=sigma,cross_sigma=cross_sigma))
inject('persistent FM lines',0,len(ts),fm&(ch%3==0),50)
for name,start,stop,spacing,phase in [('comb8',70,100,8,3),('comb4',170,200,4,1),('comb2',300,330,2,0)]:
    inject(name,start,stop,domain&(ch%spacing==phase),4)
inject('band burst 108-116',230,245,(f>=108)&(f<116),4,12)
inject('narrow line',370,400,ch==ch[np.argmin(abs(f-145))],12)
inject('broadband impulse',430,431,domain,30)
inject('cross-only change',470,480,ch==ch[np.argmin(abs(f-190))],0,20)
synth=pipeline({**synthetic,'D':injected,'cross_z':cx})
unguarded=synth['rfi_flag']  # Actual RFI evidence, never support/validity vetoes or guards.
injection_rows=[]
for e in events:
    start,stop,j=e['start'],e['stop'],e['columns']
    previous_clean=~truth_rfi[max(start-1,0),j] if start else np.zeros(len(j),bool)
    injection_rows.append(dict(event=e['name'],auto_sigma=e['sigma'],cross_sigma=e['cross_sigma'],
        first_recall=unguarded[start,j].mean(),event_recall=synth['rfi_flag'][start:stop][:,j].mean(),
        previous_row_false_fraction=unguarded[start-1,j][previous_clean].mean() if previous_clean.any() else np.nan))
injection_table=pd.DataFrame(injection_rows);display(injection_table)
clean_cells=sv&~truth_rfi;fm_gaps=clean_cells&fm[None,:]
error=(synth['model_raw']/truth-1)*rt
synthetic_metrics=dict(
    strong_coherence_null_false_flags=float(null['flag'][sv].mean()),
    weak_coherence_null_false_flags=float(weak_null['flag'][sv].mean()),
    strong_vs_weak_mask_disagreement=float((null['flag'][sv]!=weak_null['flag'][sv]).mean()),
    injected_recall=float(synth['rfi_flag'][truth_rfi].mean()),
    injected_supported_fraction=float(synth['model_support_ok'][truth_rfi].mean()),
    clean_cell_collateral=float(synth['flag'][clean_cells].mean()),
    fm_gap_retained=float((~synth['flag'][fm_gaps]).mean()),
    fm_gap_supported=float(synth['model_support_ok'][fm_gaps].mean()),
    fm_gap_model_bias_sigma=float(np.median(error[fm_gaps])),
    fm_gap_model_rmse_sigma=float(np.sqrt(np.mean(error[fm_gaps]**2))))
display(pd.Series(synthetic_metrics))

broken_mask=sv&(f[None,:]>=117.2)
broken_solver=TensorFit(ts,f,clean)
broken_model=broken_solver.solve(clean,broken_mask)
broken_infl,broken_prior,_=support(broken_solver,broken_mask)
broken_ok=(broken_infl<=P['support_inflation'])&(broken_prior<=P['prior_fraction'])&(broken_model>0)
missing=f<108
unsupported_rejection=float((~broken_ok[:,missing]).mean())
print('Deliberately unsupported 35-108 MHz rejected fraction:',unsupported_rejection)
fig,axs=plt.subplots(1,3,figsize=(15,4),sharex=True,sharey=True)
for ax,image,title in zip(axs,[truth_rfi,synth['flag'],error],
                         ['Injected RFI truth','v2 final exclusions','Background error / noise sigma']):
    im=ax.imshow(image,aspect='auto',extent=[f[0],f[-1],20,0],interpolation='nearest',
                 cmap='RdBu_r' if 'error' in title else 'gray_r',vmin=-1 if 'error' in title else 0,vmax=1)
    ax.set_title(title);ax.set_xlabel('MHz')
axs[0].set_ylabel('Minutes');fig.colorbar(im,ax=axs[2]);fig.tight_layout();plt.show()

## Acceptance and decision requested

This checkpoint fixes the unsupported-low-frequency failure only if the tests below pass. It does not declare real FM samples clean merely because the mask retains them. Cross-background drift can be astrophysical or instrumental; a change flag is conservative evidence of an unsuitable sample, not source identification. Persistent broadband RFI and a genuine continuum remain partly degenerate. Support diagnostics measure identifiable coefficients under the assumed model, not model correctness.

The method remains offline: background and scale use the training window. First-integration recall means onset localization, not causal streaming operation. Frequency dilation defaults to zero; leakage around bright lines remains a caveat. Negative counts are excluded, not unwrapped. Both the raw and public models are kept so unsupported values cannot masquerade as a recovered background.

In [ ]:
early=results['early'];af=cases['early']['f'];low=(af>=35)&(af<88)
all_runs=[*results.values(),*holdout_runs.values(),null,weak_null,synth]
checks=dict(
    solver_convergence=all(x['info']==0 for r in all_runs for x in r['solver'].history),
    cross_background_convergence=all(r['cross_stats']['reached_step_tolerance'] for r in all_runs),
    auto_robust_mask_convergence=all(r['robust_trace'].iloc[-1].changed_fraction<P['mask_change_tol'] for r in all_runs),
    early_low_band_model_supported_ge_95pct=float(early['model_support_ok'][:,low].mean())>=.95,
    early_low_band_fit_support_ge_20pct=float(early['fit_keep'][:,low].mean())>=.20,
    finite_positive_public_models=all(np.all(np.isfinite(r['model'][r['model_support_ok']]) & (r['model'][r['model_support_ok']]>0)) for r in all_runs),
    unsupported_extrapolation_rejected_ge_95pct=unsupported_rejection>=.95,
    strong_coherent_null_false_flags_le_1pct=synthetic_metrics['strong_coherence_null_false_flags']<=.01,
    coherence_background_mask_disagreement_le_1pct=synthetic_metrics['strong_vs_weak_mask_disagreement']<=.01,
    injected_first_integration_recall_ge_90pct=bool((injection_table.first_recall>=.9).all()),
    synthetic_fm_gap_retention_ge_90pct=synthetic_metrics['fm_gap_retained']>=.9,
    synthetic_fm_model_rmse_le_1sigma=synthetic_metrics['fm_gap_model_rmse_sigma']<=1.)
display(pd.Series(checks,name='acceptance'))
for name,a in cases.items():
    r=results[name];low=(a['f']>=35)&(a['f']<88);fm=(a['f']>=88)&(a['f']<108)
    print(f"{name}: low-band support {r['model_support_ok'][:,low].mean():.1%}, low-band fit {r['fit_keep'][:,low].mean():.1%}, low-band retained {(~r['flag'][:,low]).mean():.1%}; FM support {r['model_support_ok'][:,fm].mean():.1%}, FM retained {(~r['flag'][:,fm]).mean():.1%}; runtime {r['seconds']:.2f} s")
print('Checkpoint passed.' if all(checks.values()) else 'Some acceptance checks failed; inspect above before use.')
print('Real FM cleanliness and background bias remain unvalidated; support rejection is deliberate.')
# Restore primary aliases after diagnostic loop variables.
a=cases[PRIMARY];result=results[PRIMARY]
D,G,X,t,f,channel,sqrt_n=[a[k] for k in ('D','G','X','t','f','channel','rt')]
flag_v2=result['flag'];rfi_flag_v2=result['rfi_flag'];model_v2=result['model']
model_support_ok=result['model_support_ok'];fit_keep_v2=result['fit_keep']

**Review requested:** assess whether the recovered low-frequency background and the explicit support veto resolve the reported failure. Decide separately whether to pursue real FM validation or broaden to additional observing states. No new campaign stage is started here.

**Deferred findings:** calibrate the empirical residual scales and support limits; compare continuum bandwidths; test walking/non-grid-locked combs and frequency guards; distinguish changes in coherent sky/pointing from interference; use independent reference data to test real FM bias. This notebook makes no claim of an accepted campaign flag product.

**STOPPED AT REVIEW GATE — awaiting Aaron's approval.**